In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create interface widgets
path_input = widgets.Text(
    value='data.csv',
    description='File path:',
    style={'description_width': 'initial'}
)
load_button = widgets.Button(
    description='Load Data',
    button_style='success'
)
output = widgets.Output()

# Global variable to store the DataFrame
df = None

def on_load_clicked(b):
    global df
    with output:
        clear_output()
        file_path = path_input.value.strip()

        try:
            # Read CSV file
            df = pd.read_csv(file_path)

            if df.empty:
                print("⚠️ Error: File is empty!")
                return

            print("✅ Data successfully loaded!")
            display(df.head()) # Show the first rows

        except FileNotFoundError:
            print("❌ Error: File not found. Check the file path.")
        except pd.errors.EmptyDataError:
            print("❌ Error: Cannot read the file (empty or corrupted).")
        except Exception as e:
            print(f"❌ Unexpected error occurred: {e}")

load_button.on_click(on_load_clicked)

# Display widgets on the screen
display(widgets.VBox([path_input, load_button, output]))

In [ ]:
import matplotlib.pyplot as plt

if 'df' in globals() and df is not None:
    # Select only numeric columns for the plot
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()

    if numeric_cols:
        # Control widgets
        col_dropdown = widgets.Dropdown(options=numeric_cols, description='Column:')
        linewidth_slider = widgets.IntSlider(value=2, min=1, max=5, description='Line width:')

        max_idx = len(df) - 1
        range_slider = widgets.IntRangeSlider(
            value=[0, min(50, max_idx)],
            min=0, max=max_idx, step=1,
            description='Indices:'
        )

        stats_output = widgets.Output()

        def update_plot(column, linewidth, index_range):
            start, end = index_range
            subset = df[column].iloc[start:end+1]

            # Calculate statistics for the selected range
            count = subset.count()
            min_val = subset.min()
            max_val = subset.max()
            mean_val = subset.mean()
            median_val = subset.median()

            with stats_output:
                clear_output(wait=True)
                print(f"📊 Statistics for '{column}' (rows {start}–{end}):")
                print(f"• Count: {count}")
                print(f"• Min: {min_val} | Max: {max_val}")
                print(f"• Mean: {mean_val:.2f} | Median: {median_val:.2f}")

            # Draw a smooth line plot with markers
            plt.figure(figsize=(10, 4))
            plt.plot(subset.index, subset.values, marker='o', linestyle='-', linewidth=linewidth, color='blue')
            plt.title(f"Parameter Plot: {column}")
            plt.xlabel("Row Index")
            plt.ylabel(column)
            plt.grid(True)
            plt.show()

        # Link widgets to the plot update function
        widgets.interact(update_plot, column=col_dropdown, linewidth=linewidth_slider, index_range=range_slider)
        display(stats_output)
    else:
        print("⚠️ No numeric columns found in the loaded file for plotting.")
else:
    print("⚠️ Please load data in the first cell first!")